In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model inside a `while True` loop.

After each model response, the code checks whether there were any `function_call` items:

- if there are function calls, it runs the tools, appends the tool outputs to the message history, and loops again
- if there are no function calls, it breaks out of the loop

So the stopping condition is simply: **no function calls in the latest response**.


To initialize the Tracer we use this code

```bash
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")
```

To do it only once, I have placed this code in the tracer-provider-init.py, and importing this file, it will be executed only once

In [2]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps a `while True` loop that:

1. calls the model,
2. checks whether the response contains any `function_call`,
3. runs those tools and appends the outputs to `messages`,
4. calls the model again with the updated history.

It stops when `has_function_calls` stays `False` for a turn, then it `break`s out of the loop.


Q4. Saving traces to SQLite
--

In [1]:
import sqlite3
db_path="traces.db"
conn = sqlite3.connect(db_path)

In [3]:
results = conn.execute(
    """
    SELECT name, start_time, end_time, input_tokens, output_tokens, cost
    FROM spans
    """
).fetchall()

results

[('search', 1784577894529649160, 1784577894531933020, None, None, None),
 ('llm', 1784577894534283468, 1784577898228627989, 7111, 88, None),
 ('rag', 1784577894529573214, 1784577898231244595, None, None, None)]

In [7]:
for row in results:
    name, start_time, end_time, input_tokens, output_tokens, cost = row
    duration_ms = (end_time - start_time) / 1_000_000
    print(f'{name}: duration_ms: {duration_ms}, input_tokens: {input_tokens}, output_tokens: {output_tokens}, cost: {cost}')

search: duration_ms: 2.28386, input_tokens: None, output_tokens: None, cost: None
llm: duration_ms: 3694.344521, input_tokens: 7111, output_tokens: 88, cost: None
rag: duration_ms: 3701.671381, input_tokens: None, output_tokens: None, cost: None


In [9]:
results = conn.execute(
    """
    SELECT name, start_time, end_time, input_tokens, output_tokens, cost
    FROM spans
    """
).fetchall()

ron_number = 0
for row in results:
    ron_number = ron_number + 1
    name, start_time, end_time, input_tokens, output_tokens, cost = row
    duration_ms = (end_time - start_time) / 1_000_000
    print(f'{ron_number}. {name}: duration_ms: {duration_ms}, input_tokens: {input_tokens}, output_tokens: {output_tokens}, cost: {cost}')

1. search: duration_ms: 2.28386, input_tokens: None, output_tokens: None, cost: None
2. llm: duration_ms: 3694.344521, input_tokens: 7111, output_tokens: 88, cost: None
3. rag: duration_ms: 3701.671381, input_tokens: None, output_tokens: None, cost: None
4. search: duration_ms: 3.677245, input_tokens: None, output_tokens: None, cost: None
5. llm: duration_ms: 2975.35499, input_tokens: 7111, output_tokens: 96, cost: None
6. rag: duration_ms: 2984.521834, input_tokens: None, output_tokens: None, cost: None
7. search: duration_ms: 1.82233, input_tokens: None, output_tokens: None, cost: None
8. llm: duration_ms: 2331.996138, input_tokens: 7111, output_tokens: 106, cost: None
9. rag: duration_ms: 2338.461008, input_tokens: None, output_tokens: None, cost: None
10. search: duration_ms: 1.787861, input_tokens: None, output_tokens: None, cost: None
11. llm: duration_ms: 2324.460375, input_tokens: 7111, output_tokens: 121, cost: None
12. rag: duration_ms: 2331.411444, input_tokens: None, output